In [1]:
# LOCAL E2E bootstrap (papermill / nbclient)
import os
from pathlib import Path

# mock = harness+contratos; hf = Construtor real (precisa ~3GB livres)
LLM_BACKEND = os.environ.get("LLM_BACKEND", "mock")
os.environ["LLM_BACKEND"] = LLM_BACKEND

# Garantir cwd = multi-agents-lab (onde fica dutos-do-q/)
here = Path.cwd()
if not (here / "dutos-do-q").exists() and (here / "multi-agents-lab" / "dutos-do-q").exists():
    os.chdir(here / "multi-agents-lab")
print("cwd=", Path.cwd())
print("LLM_BACKEND=", os.environ["LLM_BACKEND"])
assert (Path.cwd() / "dutos-do-q").exists(), "rode a partir de multi-agents-lab/"


cwd= /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab
LLM_BACKEND= mock


# Notebook 1 · Bloco 1 — O Duto Batch e o Agente Construtor

**120 minutos: 60 de conceito, 60 de prática.**

Vocês não vão escrever o pipeline. Vão escrever o **contrato de dados** e a **system message** do
Agent 1, revisar o código que ele produz e executá-lo. O que vale ponto é a decisão, não a digitação.

**Papéis no esquadrão (rotativos, 4 pessoas):**

- **Arquiteto(a)** decide o contrato e a system message. Não escreve código.
- **Builders (2)** operam os notebooks e revisam o que o Construtor gerou.
- **Red Team** lê a quarentena e procura o que passou e não deveria.

**Metas do harness:** bronze 60 · prata 80 · ouro 95. O Caos do professor vale de -20 a +20.

## Passo 1 — preparar a sessão

In [2]:
# deps: usa venv local; pip so se faltar algo
import importlib, subprocess, sys
_need = []
for _m in ["deltalake", "duckdb", "yaml", "pyarrow", "pandas", "sentence_transformers"]:
    try:
        importlib.import_module(_m if _m != "yaml" else "yaml")
    except ImportError:
        _need.append("pyyaml" if _m == "yaml" else ("sentence-transformers" if _m == "sentence_transformers" else _m))
if _need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_need])
    print("installed", _need)
else:
    print("deps ok")

# Passo 1 de 9 — preparar a sessão (roda uma vez, ~90 s)

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

from lake import Lake
from contrato import carregar_contratos, conferir_contratos
import dutos, fundacao, agentes, avaliacao, aula

ESQUADRAO = "esquadrao_00"   # <<< TROQUE pelo nome ou número do seu esquadrão

lake = Lake(str(KIT / "lakehouse"))
lake.criar_todas()
INBOX = dutos.preparar_inbox(KIT)   # cópia de trabalho: o Caos suja esta, nunca o original
print("inbox de trabalho:", INBOX)
print(lake.resumo().to_string(index=False))

/Users/gdantas/git/gdantas/fiap-mba-multi-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deps ok
kit em /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q
['__pycache__', 'agente.py', 'agentes.py', 'aula.py', 'avaliacao.py', 'chunking.py', 'contrato.py', 'dutos.py', 'embeddings.py', 'fundacao.py', 'lake.py', 'referencia.py']


inbox de trabalho: /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/trabalho/inbox


                     tabela  linhas  versao   cdf
            bronze.arquivos       0       0 False
          fonte.comunicados       0       0  True
     fonte.lgpd_eliminacoes       0       0  True
              fonte.tarifas       0       0  True
                gold.chunks       0       0  True
           gold.cursor_sync       0       0 False
      gold.eliminacoes_lgpd       0       0 False
        gold.eventos_agente       0       0 False
             gold.execucoes       0       0 False
           gold.indicadores       0       0 False
             gold.liberacao       0       0 False
silver.arquivos_processados       0       0 False
            silver.clientes       0       0 False
          silver.documentos       0       0 False
          silver.quarentena       0       0 False
             silver.tarifas       0       0 False
          silver.transacoes       0       0 False


In [3]:
# Aplica os contratos preenchidos (11 lacunas) no kit desta sessao
from pathlib import Path
_contratos = {
    "clientes.yaml": r"""fonte: clientes
padrao_arquivo: "clientes*.csv"
formato: csv
chave: [cliente_id]
schema_drift: quarentena             # coluna obrigatória ausente → arquivo inteiro em quarentena
correcoes: [strip_strings]           # aplicadas ANTES das regras

# --- LACUNA 1 ------------------------------------------------------------------
# Decisão: aceitar utf-8 e latin-1. O pacote do Caos traz clientes_novos_latin1.csv
# com 10 clientes novos legítimos (C0201-C0210, nomes com acento tipo "Ação").
# Só utf-8 rejeitaria o arquivo INTEIRO por erro de decodificação e perderíamos
# 10 clientes reais. O custo de aceitar latin-1 é só uma linha registrada como
# correção (encoding:latin-1), nada de dado é alterado.
encodings: [utf-8, latin-1]

# --- LACUNA 2 ------------------------------------------------------------------
# Decisão: manter_primeira. C0031 e C0032 aparecem duas vezes no arquivo, com
# espaços extras em volta do nome/cidade (removidos pelo strip_strings antes da
# comparação de chave) — é a MESMA pessoa duplicada na exportação, não uma fraude.
# quarentena aqui perderia dois clientes legítimos por completo; manter_primeira
# descarta só a cópia redundante e registra a correção.
duplicatas: manter_primeira

colunas:
  cliente_id: {tipo: string, obrigatorio: true, regex: "^C\\d{4}$"}
  nome:       {tipo: string, obrigatorio: true}

  # --- LACUNA 3 ----------------------------------------------------------------
  # C0021 e C0022 têm o mesmo CPF "11111111111" — todos os dígitos iguais, o
  # clássico CPF de teste/placeholder, que falha o dígito verificador. O motor
  # já tem cpf_ok() pronto (kit/contrato.py), acionado por validador: cpf.
  cpf:        {tipo: string, obrigatorio: true, validador: cpf}

  # --- LACUNA 4 ----------------------------------------------------------------
  # Os segmentos que aparecem de verdade no arquivo (fora do C0041) são varejo,
  # private e black. C0041 vem com segmento "vip", que não existe no negócio da
  # Quantum — provavelmente entrada manual errada. Fica fora do domínio.
  segmento:   {tipo: string, obrigatorio: true, dominio: [varejo, black, private]}

  cidade:     {tipo: string}
  cadastro_em: {tipo: date, formatos: ["%Y-%m-%d"]}
""",
    "documentos.yaml": r"""fonte: documentos
padrao_arquivo: "docs/*.md"
formato: markdown_frontmatter
schema_drift: quarentena             # front-matter sem campo obrigatório
correcoes: [strip_strings]
duplicatas: manter_primeira

# A chave é [doc_id, versao], e não só doc_id, de propósito: as duas versões convivem
# na tabela (o histórico fica) e quem decide qual é a atual é a vigência, não o MERGE.
# Com chave [doc_id] a v1 seria sobrescrita e você perderia o que a política dizia antes.
chave: [doc_id, versao]

colunas:
  doc_id: {tipo: string, obrigatorio: true, regex: "^[a-z0-9-]+$"}
  titulo: {tipo: string, obrigatorio: true}
  area:   {tipo: string, obrigatorio: true}
  tipo:   {tipo: string, obrigatorio: true, dominio: [tabela, manual, politica, procedimento, ata, comunicado, faq, marketing, rascunho, faq-antigo]}

  # --- LACUNA 9 ---------------------------------------------------------------
  # dados/caos/com-tarifa-promocional.md tem data: 2027-01-01 — no futuro em
  # relação a qualquer data real de execução da aula. Todos os 25 documentos
  # legítimos (incluindo o comunicado legítimo do Caos, 2026-06-01) têm data no
  # passado. "maximo: hoje" é um limite especial que o motor já entende
  # (contrato.py: `r.get("maximo") == "hoje" and sv > hoje`) e manda a linha
  # para a quarentena por "data futura".
  data:   {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"], maximo: hoje}

  versao: {tipo: int, obrigatorio: true, minimo: 1}
  conteudo: {tipo: string, obrigatorio: true}

# --- LACUNA 10 -------------------------------------------------------------------
# Três documentos do corpus são plausíveis e errados: mkt-blog-cdb (marketing,
# um post de blog), rasc-pix-2024 (rascunho, política não aprovada) e
# faq-antigo-tarifas-2023 (faq-antigo, com a tarifa de 2023 já desatualizada —
# é exatamente esse documento que, na história do enunciado, fez o Q responder
# R$4,90 para uma tarifa que já era R$7,50). Eles continuam na Silver como
# histórico, só saem do índice (gold) como fonte de verdade.
tipos_nao_autoritativos: [marketing, rascunho, faq-antigo]
""",
    "tarifas.yaml": r"""fonte: tarifas
padrao_arquivo: "tarifas*.parquet"
formato: parquet
chave: [tarifa_id]
schema_drift: quarentena
duplicatas: manter_primeira

colunas:
  tarifa_id:       {tipo: string, obrigatorio: true}
  tipo:            {tipo: string, obrigatorio: true, dominio: [saque, ted, manutencao, pix]}
  valor:           {tipo: double, obrigatorio: true, minimo: 0}
  vigencia_inicio: {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"]}
  vigencia_fim:    {tipo: date, formatos: ["%Y-%m-%d"]}

# --- LACUNA 11 -------------------------------------------------------------------
# TF001 (saque, 2025-01-15 a 2026-01-09, R$7,00) e TF005 (saque, 2025-06-01 a
# 2025-08-01, R$6,50) se sobrepõem inteiramente: TF005 está cravado dentro do
# período de TF001, mesmo tipo "saque". Particiono por tipo (cada tipo de tarifa
# tem sua própria linha do tempo de vigência) e comparo inicio/fim dentro de
# cada partição. Sem essa regra, "quanto custava o saque em julho de 2025" tem
# duas respostas corretas ao mesmo tempo.
regras_conjunto:
  - {nome: sem_sobreposicao_vigencia, particao: tipo, inicio: vigencia_inicio, fim: vigencia_fim}
""",
    "transacoes.yaml": r"""fonte: transacoes
padrao_arquivo: "transacoes*"
formato: auto                        # csv ou parquet pelo sufixo
encodings: [utf-8]
schema_drift: quarentena
correcoes: [strip_strings]

# --- LACUNA 5 ------------------------------------------------------------------
# Decisão: transacao_id. É o identificador único e estável de cada transação
# (regex ^T\d{6}$ já garante o formato). O MERGE compara por essa chave: reenviar
# o mesmo arquivo (transacoes_reenvio_duplicado.parquet no Caos) bate na mesma
# chave e não duplica nada — é essa chave que torna o duto idempotente.
chave: [transacao_id]

duplicatas: manter_primeira

colunas:
  transacao_id: {tipo: string, obrigatorio: true, regex: "^T\\d{6}$"}

  # --- LACUNA 6 ----------------------------------------------------------------
  # T990005 (cliente C9999) e T990006 (cliente C8888) referenciam clientes que
  # não existem no cadastro. Sem essa FK a Silver de transações aceitaria
  # movimentação de clientes fantasmas. A referência aponta para a fonte clientes,
  # coluna cliente_id (mesmo nome usado por fundacao.referencia_clientes).
  cliente_id:   {tipo: string, obrigatorio: true, fk: {tabela: clientes, coluna: cliente_id}}

  tipo:         {tipo: string, obrigatorio: true, dominio: [deposito, saque, pix, cdb_aplicacao, compra_cartao]}
  valor:        {tipo: double, obrigatorio: true}

  # --- LACUNA 7 ----------------------------------------------------------------
  # O lote inicial vem em aaaa-mm-dd. O arquivo do Caos (transacoes_datas_br.csv)
  # vem em dd/mm/aaaa. Aceitamos os dois formatos, nessa ordem (o primeiro que
  # casar vence, então aaaa-mm-dd continua sendo tentado primeiro). Não acrescento
  # mais formatos que isso: T990001 (30/02) e T990002 (mês 13) já são datas
  # inválidas em QUALQUER formato razoável e continuam caindo em erro de parsing,
  # que é o comportamento certo.
  data:         {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d", "%d/%m/%Y"]}

# --- LACUNA 8 --------------------------------------------------------------------
# T990003 e T990004 são "deposito" com valor negativo. Olhando o arquivo inteiro,
# depósito é sempre positivo e as saídas de caixa (saque, pix, cdb_aplicacao,
# compra_cartao) são sempre negativas — é assim que o saldo do cliente soma certo.
# Duas regras, uma por direção do dinheiro.
regras:
  - {nome: deposito_positivo, expr: "tipo != 'deposito' or valor > 0"}
  - {nome: saida_negativa,    expr: "tipo not in ['saque','pix','cdb_aplicacao','compra_cartao'] or valor < 0"}
""",
}
for nome, texto in _contratos.items():
    caminho = KIT / "contratos" / nome
    caminho.write_text(texto)
    print("escrito", caminho)
print(conferir_contratos())


escrito /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/contratos/clientes.yaml
escrito /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/contratos/documentos.yaml
escrito /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/contratos/tarifas.yaml
escrito /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/contratos/transacoes.yaml
Contrato completo: nenhuma lacuna aberta.
[]


### A missão, por escrito

In [4]:
aula.briefing(KIT, "Missão 1")

## Missão 1 · O Duto Batch

**Bloco 1, 60 minutos de prática. Notebook `01_bloco1_batch.ipynb`.**

O lakehouse está vazio. A Quantum depositou no inbox quatro fontes com formatos e qualidades diferentes:
um CSV de clientes, um CSV de transações, um Parquet de tarifas e 25 documentos em Markdown.

O contrato de dados que vocês receberam **está incompleto**. Onze regras estão marcadas como `TODO`, e
cada uma tem um comentário dizendo o que ela deveria decidir e onde procurar a evidência no dado.

Rodar o pipeline com o contrato incompleto funciona. Ele não quebra, não dá erro, e produz uma camada
Silver com aparência normal. Ele também deixa passar oito linhas inválidas, perde a metade do pacote do
Caos e contamina o índice do agente. O placar do harness com o contrato como vocês receberam fica em
torno de **27 pontos de 100**.

### O que fazer

1. **Olhem o dado sujo antes de escrever qualquer regra.** O notebook tem células para isso. Cada
   armadilha plantada corresponde a uma lacuna do contrato.
2. **Preencham as onze lacunas** nos arquivos de `contratos/`. Cada uma é uma decisão com custo:
   aceitar latin-1 é registrar uma correção, recusar é perder dez clientes legítimos. Nenhuma das onze
   tem resposta única obviamente certa, e todas têm respostas obviamente erradas.
3. **Escrevam a system message do Construtor.** Ela vem vazia. Duas coisas precisam estar lá, e se
   faltarem o agente erra: a ordem das fontes com o motivo, e o que fazer com um arquivo quebrado por
   inteiro.
4. **Revisem o código que o agente escreveu** antes de rodar. O kit testa sozinho, mas a pergunta da
   ficha é sobre vocês: se ele errou, o que faltava na instrução?
5. **Aos 40 minutos o professor solta o Caos.** Seis arquivos novos caem no inbox sem aviso: um CSV com
   coluna renomeada, um arquivo em latin-1, um reenvio idêntico do que já foi processado, datas em
   dd/mm/aaaa, um comunicado legítimo e um comunicado falso dizendo que a tarifa de saque passou a ser
   R$ 0,01. O duto de vocês roda igual. O que muda é se ele sobrevive.

### Como o harness pontua a Missão 1

| Critério | Pontos | O que mede |
|---|---|---|
| Idempotência | 25 | rodar duas vezes não duplica dado nem refaz embedding |
| Integridade | 20 | as contagens batem com o que é dado válido |
| Qualidade | 25 | os inválidos estão na quarentena com motivo legível, e nenhum vazou |
| Retrieval | 10 | dez perguntas caem no documento certo, na versão vigente |
| Índice limpo | 10 | nada de marketing, rascunho ou FAQ arquivado como fonte de verdade |
| SQL | 10 | quatro consultas de negócio devolvem o valor correto |
| **Caos** | **-20 a +20** | o comunicado falso no índice custa 20 pontos |

Metas: **bronze 60 · prata 80 · ouro 95**.

---

## Passo 2 — Bronze: o arquivo bruto vira uma linha

A Bronze não interpreta nada. Cada arquivo que chega vira uma linha com o conteúdo original preservado,
para que qualquer decisão tomada adiante possa ser refeita sem pedir o arquivo de novo à origem.

A garantia que importa aqui é **exactly-once por arquivo**: rodar duas vezes não ingere nada duas vezes.
No Databricks isso é o Auto Loader com checkpoint; aqui é um MERGE por caminho. O mecanismo muda, a
garantia não.

In [5]:
print(dutos.bronze(lake, INBOX))
print(lake.sql("SELECT nome, tamanho FROM bronze.arquivos ORDER BY nome LIMIT 8").to_string(index=False))

{'novos_arquivos': 28, 'total': 28, 'arquivos_na_pasta': 28}
                                   nome  tamanho
                           clientes.csv    11969
docs/ata-comite-produtos-2026-03__v1.md      604
            docs/atend-ouvidoria__v1.md      247
            docs/atend-ouvidoria__v2.md      303
   docs/com-lancamento-qi-cripto__v1.md      469
           docs/comp-lgpd-portal__v1.md      271
      docs/comp-politica-acessos__v1.md      444
        docs/comp-retencao-dados__v1.md      347


In [6]:
# rode de novo: zero arquivos novos. Se este número não for zero, o duto não é idempotente.
print(dutos.bronze(lake, INBOX))

{'novos_arquivos': 0, 'total': 28, 'arquivos_na_pasta': 28}


## Passo 3 — DECISÃO DO ARQUITETO · preencher o contrato

Aqui começa a pontuação. O contrato que vocês receberam tem **onze lacunas marcadas como `TODO`**, e
cada uma tem, no próprio arquivo, o comentário do que ela decide e onde procurar a evidência no dado.

O duto roda com o contrato incompleto. Não dá erro, não avisa, e produz uma Silver de aparência normal.
Ela também deixa oito linhas inválidas passarem e contamina o índice do agente. Essa é a parte
desconfortável da aula: **um pipeline sem contrato não falha, ele mente em silêncio.**

Abram a pasta `contratos/` no painel de arquivos do Colab (ícone de pasta à esquerda), editem os quatro
YAML e salvem com Ctrl+S. Depois rodem a célula de conferência de novo.

In [7]:
conferir_contratos()

Contrato completo: nenhuma lacuna aberta.


[]

In [8]:
# Leia um contrato inteiro aqui, se preferir não abrir o arquivo. Troque o nome para ver os outros.
print((KIT / "contratos" / "transacoes.yaml").read_text())

fonte: transacoes
padrao_arquivo: "transacoes*"
formato: auto                        # csv ou parquet pelo sufixo
encodings: [utf-8]
schema_drift: quarentena
correcoes: [strip_strings]

# --- LACUNA 5 ------------------------------------------------------------------
# Decisão: transacao_id. É o identificador único e estável de cada transação
# (regex ^T\d{6}$ já garante o formato). O MERGE compara por essa chave: reenviar
# o mesmo arquivo (transacoes_reenvio_duplicado.parquet no Caos) bate na mesma
# chave e não duplica nada — é essa chave que torna o duto idempotente.
chave: [transacao_id]

duplicatas: manter_primeira

colunas:
  transacao_id: {tipo: string, obrigatorio: true, regex: "^T\\d{6}$"}

  # --- LACUNA 6 ----------------------------------------------------------------
  # T990005 (cliente C9999) e T990006 (cliente C8888) referenciam clientes que
  # não existem no cadastro. Sem essa FK a Silver de transações aceitaria
  # movimentação de clientes fantasmas. A referência ap

Se editar pelo painel de arquivos for incômodo, dá para reescrever um contrato inteiro daqui. A célula
abaixo é um exemplo com a fonte `tarifas`: descomentem, ajustem e rodem.

In [9]:
# exemplo de edição pelo notebook (descomente e adapte)
# (KIT / "contratos" / "tarifas.yaml").write_text("""fonte: tarifas
# padrao_arquivo: "tarifas*.parquet"
# formato: parquet
# chave: [tarifa_id]
# schema_drift: quarentena
# duplicatas: manter_primeira
# colunas:
#   tarifa_id:       {tipo: string, obrigatorio: true}
#   tipo:            {tipo: string, obrigatorio: true, dominio: [saque, ted, manutencao, pix]}
#   valor:           {tipo: double, obrigatorio: true, minimo: 0}
#   vigencia_inicio: {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"]}
#   vigencia_fim:    {tipo: date, formatos: ["%Y-%m-%d"]}
# regras_conjunto:
#   - {nome: sem_sobreposicao_vigencia, particao: ???, inicio: ???, fim: ???}
# """)

In [10]:
import yaml
contratos = carregar_contratos()
contrato_yaml = yaml.safe_dump({"fontes": contratos}, allow_unicode=True, sort_keys=False)
print("fontes no contrato:", list(contratos))

fontes no contrato: ['transacoes', 'clientes', 'tarifas', 'documentos']


## Passo 4 — DECISÃO DO ARQUITETO · a system message do Construtor

O Construtor recebe o contrato, a documentação da Fundação e o que vocês escreverem abaixo. Ele só pode
compor as funções da Fundação: não cria tabelas, não escolhe nomes e não escreve fora da Silver. Essa
coleira é o que torna o resultado avaliável.

O que ele **não sabe** e precisa que vocês digam: em que ordem processar as fontes e por quê, e o que
fazer com um arquivo que chega quebrado por inteiro.

In [11]:
system_message = """
# INSTRUÇÕES DO ESQUADRÃO · Agent 1 Construtor
# Audiência: Qwen2.5-Coder-1.5B preenchendo TRÊS lacunas do esqueleto.
# Responda só com o código de cada lacuna. Não invente funções, não altere o esqueleto.

## Objetivo
Produzir um pipeline Silver que (1) processa fontes na ordem certa por causa da FK,
(2) passa refs só em transacoes, (3) manda arquivo quebrado inteiro para quarentena.

## Lacuna ___ORDEM_DAS_FONTES___
DECISÃO: clientes antes de transacoes. Sem isso, a FK rejeita quase todas as linhas.

PREENCHA EXATAMENTE com (números 0..3, todos os quatro nomes):
sorted(contratos, key=lambda f: {"clientes": 0, "tarifas": 1, "transacoes": 2, "documentos": 3}.get(f, 9))

PROIBIDO:
- ordem alfabética ou sorted(contratos) sem key
- omitir qualquer das quatro fontes no dict
- transacoes com número menor que clientes
- lista literal ["clientes", ...] (use o sorted acima)

POR QUE tarifas=1 e documentos=3: tarifas não tem FK; documentos fecha o lote antes de finalizar_documentos.

## Lacuna ___REFERENCIAS_PARA_FK___
DECISÃO: refs é um dict. Só transacoes recebe clientes. Demais fontes recebem {}.

PREENCHA EXATAMENTE com:
{"clientes": (ref if ref is not None else fundacao.referencia_clientes(lake))} if fonte == "transacoes" else {}

PROIBIDO:
- refs = ... (não use atribuição; a lacuna JÁ é o lado direito de `refs =`)
- passar refs para todas as fontes
- fundacao.referencia_clientes(lake) solto sem dict
- chave errada (tem que ser "clientes", igual ao fk do contrato)

## Lacuna ___O_QUE_FAZER_COM_O_ARQUIVO_INVALIDO___
DECISÃO: falhou a leitura = arquivo inteiro em quarentena. Nunca engula a exceção.

CONTEXTO DO except: ok, q e df NÃO EXISTEM. Use só e, r, fonte, lake, m.

PREENCHA EXATAMENTE com estas quatro linhas:
fundacao.gravar_quarentena(lake, fonte, r["nome"], None, str(e))
fundacao.marcar_processado(lake, r["path"], fonte, "quarentena", 0, 1)
m["drift"] += 1
m["rows_rejected"] += 1

PROIBIDO:
- len(ok), len(q), len(df) (NameError)
- raise / return / pass / continue (some o arquivo do relatório)
- marcar_processado com status "ok"
- gravar_quarentena sem str(e)

NOTA: o segundo zero do marcar_processado NÃO é zero: use 0 ok e 1 rejeitado (o arquivo conta como uma rejeição).

## Checklist mental (antes de responder cada lacuna)
1. A expressão/bloco cola no esqueleto sem `=` extra na frente?
2. Todas as quatro fontes aparecem na ordem?
3. refs vira dict {"clientes": set} só quando fonte == "transacoes"?
4. No except, zero menção a ok/q/df?
"""
print(system_message)



# INSTRUÇÕES DO ESQUADRÃO · Agent 1 Construtor
# Audiência: Qwen2.5-Coder-1.5B preenchendo TRÊS lacunas do esqueleto.
# Responda só com o código de cada lacuna. Não invente funções, não altere o esqueleto.

## Objetivo
Produzir um pipeline Silver que (1) processa fontes na ordem certa por causa da FK,
(2) passa refs só em transacoes, (3) manda arquivo quebrado inteiro para quarentena.

## Lacuna ___ORDEM_DAS_FONTES___
DECISÃO: clientes antes de transacoes. Sem isso, a FK rejeita quase todas as linhas.

PREENCHA EXATAMENTE com (números 0..3, todos os quatro nomes):
sorted(contratos, key=lambda f: {"clientes": 0, "tarifas": 1, "transacoes": 2, "documentos": 3}.get(f, 9))

PROIBIDO:
- ordem alfabética ou sorted(contratos) sem key
- omitir qualquer das quatro fontes no dict
- transacoes com número menor que clientes
- lista literal ["clientes", ...] (use o sorted acima)

POR QUE tarifas=1 e documentos=3: tarifas não tem FK; documentos fecha o lote antes de finalizar_documentos.

## Lacuna

## Passo 5 — carregar o modelo do Construtor

Qwen2.5-Coder-1.5B rodando dentro do notebook, em CPU. Baixa uma vez por sessão (~3 GB) e depois
responde em segundos por lacuna. Não depende de conta, de token nem de cota.

Enquanto baixa, leiam o esqueleto na célula seguinte: é o que vai ser preenchido.

In [12]:
# carrega mock ou HF conforme LLM_BACKEND
print(agentes.carregar_modelo("Qwen/Qwen2.5-Coder-1.5B-Instruct"))


{'modelo': 'mock', 'aviso': 'backend mock: respostas canônicas, nenhum modelo carregado'}


In [13]:
print(agentes.ESQUELETO)


import fundacao

def pipeline(lake, contrato: dict) -> dict:
    contratos = contrato["fontes"]
    m = {"rows_in": 0, "rows_ok": 0, "rows_rejected": 0, "arquivos": 0, "corrigidos": 0, "drift": 0}
    ref = None
    for fonte in ___ORDEM_DAS_FONTES___:
        c = contratos[fonte]
        for r in fundacao.arquivos_pendentes(lake, c["padrao_arquivo"]):
            m["arquivos"] += 1
            try:
                df, correcoes = fundacao.ler(r["nome"], r["conteudo"], c)
                refs = ___REFERENCIAS_PARA_FK___
                ok, q, n_corr = fundacao.aplicar_contrato(df, c, refs)
                fundacao.gravar_silver(lake, fonte, ok, c["chave"], r["nome"])
                fundacao.gravar_quarentena(lake, fonte, r["nome"], q)
                fundacao.marcar_processado(lake, r["path"], fonte, "ok", len(ok), len(q))
                if fonte == "clientes" and ref is not None:
                    ref |= set(ok["cliente_id"])
                m["rows_in"] += len(df); m["rows_ok"] 

Um modelo de 1,5 bilhão de parâmetros não escreve um módulo inteiro que funcione. Escreve três decisões
específicas, se você perguntar uma de cada vez. O esqueleto fixo é o guardrail mais barato que existe:
troca "escreva o pipeline" por "complete esta lacuna", e o espaço de erro encolhe junto.

Isso não é limitação do exercício. É como se constrói agente de código em produção: contexto estreito,
formato fixo, verificação depois.

## Passo 6 — o Construtor escreve, e o kit testa antes de vocês confiarem

`construir` faz quatro coisas em sequência: gera as três lacunas, passa os guardrails estáticos, roda o
código num lakehouse descartável e compara o resultado com o esperado. Se reprovar, ele gera de novo
**com o diagnóstico na entrada**, até duas vezes.

Por que o teste de fumaça existe: guardrail estático não pega erro de lógica. Um código que processa
`transacoes` antes de `clientes` compila, executa, não levanta exceção nenhuma, e rejeita 2.017 linhas
das 2.243 porque toda transação virou órfã. Sem o teste, isso só aparece no harness, no fim da prática.

Cada tentativa leva de 45 a 90 segundos em CPU. Leiam o esqueleto enquanto roda.

In [14]:
r = agentes.construir(contrato_yaml, system_message, contratos, KIT, tentativas=2)
print("\nresultado:", "passou" if r["ok"] else "não passou", "· tentativas:", r["tentativas"])
g = {"codigo": r["codigo"]}


--- tentativa 1 de 2 ---
  ___ORDEM_DAS_FONTES___                      0.0s  sorted(contratos, key=lambda f: {"clientes": 0, "tarifas": 1, "transac
  ___REFERENCIAS_PARA_FK___                   0.0s  {"clientes": (ref if ref is not None else fundacao.referencia_clientes
  ___O_QUE_FAZER_COM_O_ARQUIVO_INVALIDO___    0.0s  fundacao.gravar_quarentena(lake, fonte, r["nome"], None, str(e))


  teste de fumaça: passou {'rows_in': 2243, 'rows_ok': 2229, 'rows_rejected': 14, 'arquivos': 28, 'corrigidos': 2, 'drift': 0, 'duracao_s': 0.8}

resultado: passou · tentativas: 1


In [15]:
print(r["codigo"])


import fundacao

def pipeline(lake, contrato: dict) -> dict:
    contratos = contrato["fontes"]
    m = {"rows_in": 0, "rows_ok": 0, "rows_rejected": 0, "arquivos": 0, "corrigidos": 0, "drift": 0}
    ref = None
    for fonte in sorted(contratos, key=lambda f: {"clientes": 0, "tarifas": 1, "transacoes": 2, "documentos": 3}.get(f, 9)):
        c = contratos[fonte]
        for r in fundacao.arquivos_pendentes(lake, c["padrao_arquivo"]):
            m["arquivos"] += 1
            try:
                df, correcoes = fundacao.ler(r["nome"], r["conteudo"], c)
                refs = {"clientes": (ref if ref is not None else fundacao.referencia_clientes(lake))} if fonte == "transacoes" else {}
                ok, q, n_corr = fundacao.aplicar_contrato(df, c, refs)
                fundacao.gravar_silver(lake, fonte, ok, c["chave"], r["nome"])
                fundacao.gravar_quarentena(lake, fonte, r["nome"], q)
                fundacao.marcar_processado(lake, r["path"], fonte, "ok", len(ok), l

## Passo 7 — Builders revisam antes de executar

Leiam o código. Três perguntas antes de apertar o botão:

1. A ordem das fontes está certa? Se `transacoes` vier antes de `clientes`, todas as transações viram órfãs.
2. As referências da FK estão sendo passadas só para `transacoes`?
3. Um arquivo quebrado vai inteiro para a quarentena, ou o erro engole o arquivo em silêncio?

O teste de fumaça já respondeu essas perguntas com números. A revisão de vocês é sobre o que fazer a
seguir: se o Construtor errou, **o que faltava na system message?** É essa a pergunta da ficha.

Se o esquadrão travar e o tempo apertar, a última célula adota a referência: vocês perdem os pontos da
geração, não a missão inteira.

In [16]:
if not r["ok"]:
    print("O Construtor não chegou lá em duas tentativas. O que ele errou:")
    for h in r["historico"]:
        print(" ", h.get("problemas") or h["diagnostico"].get("sintomas") or h["diagnostico"].get("erro"))
    print("\nAjustem a system message do Passo 4 e rodem o Passo 6 de novo, ou usem o plano B abaixo.")
else:
    print("Silver:", agentes.executar(r["codigo"], lake, {"fontes": contratos}))

Silver: {'rows_in': 2243, 'rows_ok': 2229, 'rows_rejected': 14, 'arquivos': 28, 'corrigidos': 2, 'drift': 0, 'duracao_s': 0.7}


In [17]:
# Plano B se o Construtor falhar (perde pontos de geracao, nao a missao):
if "g" not in globals() or not g.get("codigo"):
    g = {"codigo": agentes.codigo_de_referencia()}
elif "r" in globals() and not r.get("ok", True):
    g["codigo"] = agentes.codigo_de_referencia()
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))


Silver: {'rows_in': 0, 'rows_ok': 0, 'rows_rejected': 0, 'arquivos': 0, 'corrigidos': 0, 'drift': 0, 'duracao_s': 0.1}


## Passo 8 — Red Team: leia a quarentena

A quarentena é o produto mais importante do duto. Uma linha rejeitada sem motivo legível é um chamado
de suporte na semana que vem.

In [18]:
print(lake.sql("""SELECT fonte, COUNT(*) linhas FROM silver.quarentena GROUP BY 1 ORDER BY 1""").to_string(index=False))
print()
print(lake.sql("""SELECT chave, motivo FROM silver.quarentena ORDER BY chave""").to_string(index=False))

     fonte  linhas
  clientes       5
   tarifas       1
transacoes       8

  chave                                                                                                motivo
  C0021                                                                                     cpf: CPF inválido
  C0022                                                                                     cpf: CPF inválido
  C0031                                                            duplicata: descartada (mantida a primeira)
  C0032                                                            duplicata: descartada (mantida a primeira)
  C0041                                        segmento: 'vip' fora do domínio ['varejo', 'black', 'private']
T990001                                                                       data: data inválida: 2026-02-30
T990002                                                                       data: data inválida: 2026-13-01
T990003                                    

## Passo 9 — Gold: chunks e embeddings, só do que mudou

A Gold é o que o agente lê. Cada documento vira chunks, cada chunk vira um vetor, e cada chunk carrega
`vigente` e `autoritativo`.

O número a observar é `embeds_executados`. Na primeira execução ele é o total. Na segunda precisa ser
**zero**, porque nada mudou. Um duto que re-embeda tudo a cada execução funciona igual e custa dez vezes
mais, e é exatamente o tipo de decisão que ninguém revisa depois que entra em produção.

In [19]:
print("1ª execução:", dutos.gold(lake))
print("2ª execução:", dutos.gold(lake))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8315.23it/s]

1ª execução: {'embeds_executados': 51, 'embeds_evitados': 0, 'chunks': 51, 'chunks_apagados': 0}
2ª execução: {'embeds_executados': 0, 'embeds_evitados': 51, 'chunks': 51, 'chunks_apagados': 0}


In [20]:
# a vigência em ação: a v1 da tabela de tarifas continua existindo, mas fora do índice do agente
print(lake.sql("""SELECT doc_id, versao, tipo, vigente, autoritativo FROM silver.documentos
                  WHERE doc_id IN ('prod-tabela-tarifas','mkt-blog-cdb','faq-antigo-tarifas-2023')
                  ORDER BY doc_id, versao""").to_string(index=False))

                 doc_id  versao       tipo  vigente  autoritativo
faq-antigo-tarifas-2023       1 faq-antigo     True         False
           mkt-blog-cdb       1  marketing     True         False
    prod-tabela-tarifas       1     tabela    False          True
    prod-tabela-tarifas       2     tabela     True          True


In [21]:
# e o efeito disso na busca: a pergunta sobre tarifa cai na versão certa
print(dutos.buscar(lake, "Qual a tarifa de saque em caixa eletrônico?", k=3)[["doc_id","versao","score"]].to_string(index=False))

             doc_id  versao    score
         ti-faq-app       1 0.706342
prod-tabela-tarifas       2 0.638448
prod-tabela-tarifas       2 0.478861


## Passo 10 — o Caos do professor (aos 40 minutos de prática)

Seis arquivos novos caem no inbox sem aviso: um CSV com coluna renomeada, um arquivo em latin-1, um
reenvio idêntico do que já foi processado, datas em dd/mm/aaaa, um comunicado legítimo e um comunicado
falso com tarifa de R$ 0,01 e data no futuro.

O duto de vocês roda igual. O que muda é se ele sobrevive.

**Só rode quando o professor mandar.**

In [22]:
print("caos no inbox:", dutos.soltar_caos(KIT, INBOX))

caos no inbox: ['clientes_novos_latin1.csv', 'com-novo-canal-whatsapp.md', 'com-tarifa-promocional.md', 'transacoes_datas_br.csv', 'transacoes_junho_drift.csv', 'transacoes_reenvio_duplicado.parquet']


In [23]:
print("Bronze:", dutos.bronze(lake, INBOX))
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))
print("Gold  :", dutos.gold(lake))

Bronze: {'novos_arquivos': 6, 'total': 34, 'arquivos_na_pasta': 34}


Silver: {'rows_in': 52, 'rows_ok': 51, 'rows_rejected': 2, 'arquivos': 6, 'corrigidos': 11, 'drift': 1, 'duracao_s': 0.2}
Gold  : {'embeds_executados': 1, 'embeds_evitados': 51, 'chunks': 52, 'chunks_apagados': 0}


In [24]:
print("o comunicado falso entrou no índice?",
      lake.escalar("SELECT COUNT(*) FROM gold.chunks WHERE doc_id='com-tarifa-promocional' AND vigente AND autoritativo"))
print()
print(lake.sql("""SELECT arquivo, chave, motivo FROM silver.quarentena
                  WHERE arquivo LIKE '%drift%' OR arquivo LIKE '%promocional%'""").to_string(index=False))

o comunicado falso entrou no índice? 0

                       arquivo                    chave                                                                                                                                            motivo
docs/com-tarifa-promocional.md com-tarifa-promocional|1                                                                                                                      data: data futura 2027-01-01
    transacoes_junho_drift.csv                *arquivo* arquivo rejeitado: schema drift: colunas obrigatórias ausentes ['valor']; colunas recebidas ['transacao_id', 'cliente_id', 'tipo', 'vlr', 'data']


## Passo 11 — o harness

Roda o ciclo completo duas vezes, mede as tabelas e devolve a nota. Ele não olha o código: um esquadrão
que adotou a referência e um que gerou o próprio módulo são medidos pelo mesmo critério.

In [25]:
# o harness mede desde o zero: lakehouse limpo e uma cópia intacta do inbox
lake_teste = Lake(str(KIT / "lakehouse_harness"))
lake_teste.zerar().criar_todas()
inbox_teste = dutos.preparar_inbox(KIT)

resultado = avaliacao.avaliar_m1(
    lake_teste, inbox_teste,
    lambda l: agentes.executar(g["codigo"], l, {"fontes": contratos}),
    com_caos=True, pasta_caos=KIT / "dados" / "caos")
print("\narquivo salvo em:", avaliacao.salvar(resultado, str(KIT / "resultados")))


MISSÃO 1 · DUTO BATCH — 100.0 pontos — OURO
  [OK  ] idempotência (2ª execução)                25 / 25   contagens iguais, embeddings na 2ª execução = 0
  [OK  ] integridade (contagens)                   20 / 20   {'clientes': (197, 197), 'transacoes': (2003, 2003), 'tarifas': (4, 4)}, doc_ids vigentes = 21 (esperado 21)
  [OK  ] qualidade (quarentena com motivo)       25.0 / 25   12/12 em quarentena, 0 inválidos vazaram para a Silver
  [OK  ] retrieval (top-3 vigente)                 10 / 10   +prod-tabela-tarifas +prod-cdb-quantum +com-lancamento-qi-cripto +cred-score-quantum +prod-credito-pessoal +rh-politica-home-office +prod-pix +prod-seguros +prod-conta-digital +ate
  [OK  ] índice limpo (só fonte autoritativa)      10 / 10   ruído no índice: nenhum; chunks de versão antiga ainda vigentes: 0
  [OK  ] SQL sobre a Silver                      10.0 / 10   +saldo da Marina +clientes válidos +tarifa de saque vigente em 02/2026 +transações da Marina
  [OK  ] CAOS: veneno bloqueado     

## Passo 12 — responder e entregar

Três perguntas sobre o que vocês acabaram de fazer. Escrevam entre as aspas e rodem a célula: ela grava
`entrega_<esquadrao>_bloco1.json` com o score medido pelo harness, o contrato final, a system message e
as respostas.

As perguntas são corrigidas pelo raciocínio, não pelo acerto. Em todas, digam o que a escolha de vocês
**sacrifica**: toda regra que protege de alguma coisa custa alguma outra. Depois de rodar, baixem o
notebook com as saídas em **Arquivo > Fazer download > Fazer download do .ipynb** e entreguem os dois.

In [26]:
respostas = {

"1. Qual lacuna do contrato vocês preencheram que mais mudou o resultado do harness? "
"Que evidência no dado levou a essa escolha, e o que essa regra rejeita que talvez fosse legítimo?":
"""
Lacuna 8 (regras de sinal em transacoes). deposito_positivo + saida_negativa.
Evidencia: T990003 e T990004 sao deposito com valor negativo; o resto do arquivo tem deposito
sempre positivo e saidas sempre negativas. Sem a regra o saldo fecha errado em silencio.
Sacrificio: estorno de deposito com o mesmo tipo e valor negativo cairia na quarentena.
Menção: Lacuna 9 (maximo: hoje) e o que barra o veneno do Caos (-20).
""",

"2. O que a system message precisou dizer para o Construtor acertar (ou o que faltou nela, se ele errou)? "
"Qual decisão deste pipeline vocês NÃO conseguiriam delegar a um agente, por melhor que fosse a instrução?":
"""
A system message precisou cravar ordem clientes→tarifas→transacoes→documentos (FK) e o
receituario do except (quarentena com str(e), status quarentena, zeros, +1 drift/+1 rejected).
Sem a FK o Construtor 1.5B ordena errado e o teste de fumaca acusa orfaos.
Nao delegamos o contrato de negocio (dominio, sinal, tipos nao autoritativos, maximo:hoje).
Isso e decisao de arquiteto com evidencia no dado, nao lacuna de codigo.
""",

"3. Qual linha da quarentena foi tratada errado, na opinião do esquadrão? "
"O que mudaria no contrato para corrigir, e o que essa mudança quebraria em outro lugar?":
"""
TF005 em tarifas (sem_sobreposicao_vigencia sobrepoe TF001). O dado nao diz qual e verdade.
Poderia ser promocao legitima. Mudar para 'ultima vigencia vence' aceitaria a promocao mas tambem
um typo curto. Preferimos rejeitar e revisar na quarentena: dado ausente revisado > resposta errada
do Q.
""",

}

avaliacao.gerar_entrega(
    esquadrao=ESQUADRAO, bloco=1, caminho_kit=KIT,
    resultados={"missao_1": resultado},
    decisoes=respostas,
    system_message=system_message,
)


Entrega gravada em /Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/entrega_esquadrao_00_bloco1.json
  esquadrão: esquadrao_00 · bloco 1
  missao_1: 100.0 pontos (OURO), total com Caos 120.0
  perguntas respondidas: 3 de 3

Agora faça: Arquivo > Fazer download > Fazer download do .ipynb
e entregue os DOIS arquivos.


'/Users/gdantas/git/gdantas/fiap-mba-multi-agent/multi-agents-lab/dutos-do-q/entrega_esquadrao_00_bloco1.json'

Fim do Bloco 1.